In [ ]:
# Conferência manual: soma os óbitos de todas as Regionais para cada ano e
# compara com o esperado, a partir da tabela consolidada de mortalidade.
import pandas as pd

df = pd.read_csv(
    r'/home/anna/tcc/Seleção e Transformação/Mortalidade/tabela_mortes_por_faixa_etária.csv'
)

# Seleciona somente as colunas de total por ano
colunas_total = [
    coluna for coluna in df.columns
    if coluna.endswith('_TOTAL')
    and not coluna.endswith('_TOTAL_QUANTIDADE')
]

# Soma todas as regionais de saúde para cada ano
totais_por_ano = df[colunas_total].apply(pd.to_numeric, errors='coerce').sum()

# Transforma em DataFrame
resultado = totais_por_ano.reset_index()
resultado.columns = ['Ano', 'Total']

# Extrai somente o ano
resultado['Ano'] = resultado['Ano'].str.replace('_TOTAL', '').astype(int)

# Ordena por ano
resultado = resultado.sort_values('Ano').reset_index(drop=True)

print(resultado)

In [3]:
REGIONAL_ALVO = "Divinópolis"
ANOS = list(range(1979, 2025))
FAIXAS_ETARIAS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70 OU MAIS"]

# ── Filtra municípios da regional ────────────────────────────────────────────
df_divinopolis = df[df["REGIONAL DE SAÚDE"].str.strip() == REGIONAL_ALVO].copy()

# ── Tabela de municípios da regional ─────────────────────────────────────────
df_municipios = df_divinopolis[["MUNICÍPIO", "CÓDIGO IBGE"]].sort_values("MUNICÍPIO").reset_index(drop=True)
print(f"Municípios da regional de {REGIONAL_ALVO}: {len(df_municipios)}")
display(df_municipios)

# ── Total por ano (soma dos municípios da regional) ──────────────────────────
colunas_total_ano = [f"{ano}_TOTAL" for ano in ANOS]
df_soma_ano = (
    df_divinopolis[colunas_total_ano]
    .sum(skipna=True)
    .rename_axis("ANO")
    .reset_index(name="ÓBITOS")
)
df_soma_ano["ANO"] = ANOS
display(df_soma_ano)

# ── Total por faixa etária (soma de todos os anos e municípios) ─────────────
df_soma_faixa = pd.DataFrame({
    "FAIXA ETÁRIA": FAIXAS_ETARIAS,
    "ÓBITOS": [
        df_divinopolis[[f"{ano}_{faixa}" for ano in ANOS]].sum(skipna=True).sum()
        for faixa in FAIXAS_ETARIAS
    ],
})
display(df_soma_faixa)

print(f"\nTotal geral ({REGIONAL_ALVO}): {int(df_soma_ano['ÓBITOS'].sum())}")

Municípios da regional de Divinópolis: 55


,MUNICÍPIO,CÓDIGO IBGE
0,aguanil,3100807
1,araujos,3103900
2,arcos,3104205
3,bambui,3105103
4,bom despacho,3107406
5,bom sucesso,3108008
6,camacho,3110400
7,campo belo,3111200
8,cana verde,3111903
9,candeias,3112000


,ANO,ÓBITOS
0,1979,24.0
1,1980,19.0
2,1981,25.0
3,1982,18.0
4,1983,25.0
5,1984,23.0
6,1985,26.0
7,1986,25.0
8,1987,39.0
9,1988,29.0


,FAIXA ETÁRIA,ÓBITOS
0,0-9,0.0
1,10-19,1.0
2,20-29,17.0
3,30-39,166.0
4,40-49,468.0
5,50-59,623.0
6,60-69,589.0
7,70 OU MAIS,820.0



Total geral (Divinópolis): 2684


In [ ]:
# ── Total por ano E por faixa etária (matriz ano x faixa) ────────────────────
# Mesma conferência anterior, mas detalhada por faixa etária em vez de só o total.
df_soma_ano_faixa = pd.DataFrame(
    {
        faixa: [df_divinopolis[f"{ano}_{faixa}"].sum(skipna=True) for ano in ANOS]
        for faixa in FAIXAS_ETARIAS
    },
    index=ANOS,
)
df_soma_ano_faixa.index.name = "ANO"
df_soma_ano_faixa["TOTAL"] = df_soma_ano_faixa.sum(axis=1)
display(df_soma_ano_faixa)

População

In [5]:
# Loaded variable 'df' from URI: /home/anna/tcc/Seleção e Transformação/População/tabela_populacao_faixa_etaria.csv
import pandas as pd
df = pd.read_csv(r'/home/anna/tcc/Seleção e Transformação/População/tabela_populacao_faixa_etaria.csv')

In [ ]:
# Confere a população total da regional de Divinópolis nos anos censitários
# chave, usada como denominador no cálculo de taxa.
REGIONAL_ALVO = "divinopolis"
ANOS_ALVO = [1970, 1980, 2022]

df_divinopolis = df[df["REGIONAL DE SAÚDE"].str.strip() == REGIONAL_ALVO].copy()

df_soma_populacao = pd.DataFrame({
    "ANO": ANOS_ALVO,
    "POPULAÇÃO": [df_divinopolis[f"{ano}_TOTAL"].sum(skipna=True) for ano in ANOS_ALVO],
})
display(df_soma_populacao)

In [ ]:
# Confere a população total do estado (todos os municípios) num ano específico.
ANO_ALVO = 1980


soma_populacao = df[f"{ANO_ALVO}_TOTAL"].sum(skipna=True)

print(f"População total em {ANO_ALVO}: {soma_populacao:,.0f}")